In [4]:
import requests
import os
import pandas as pd
from datetime import datetime
from urllib.parse import quote_plus
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [5]:
load_dotenv()
url = "https://brapi.dev/api/v2/stocks/historical#"
token = os.environ["BRAPI_TOKEN"]
hearder = {"Authorization": f"Bearer {token}"}

simbolo = ['PETR4', 'VALE3', 'MGLU3','ITUB4']

response = requests.request("GET", url, headers=hearder
, params={"symbols": ",".join(simbolo),
           "range": "2y",
           "interval": "5d",
           "sortOrder": "asc",
           }, timeout=30,)

In [6]:
resultado = response.json()
linhas = []
for i in resultado['results']:
  data= i.get('data')
  if not data:
    continue
  for dia in data['historicalDataPrice']:
    linhas.append({
      'Simbolo': i['requestedSymbol'],
      "Data": datetime.fromtimestamp(dia['date']).strftime('%d/%m/%Y'),
      "Open": dia['open'],
      "High": dia['high'],
      'Low': dia['low'],
      'Close': dia['close'],
    })

In [7]:
resultado

{'results': [{'requestedSymbol': 'PETR4',
   'symbol': 'PETR4',
   'changed': False,
   'data': {'usedInterval': '5d',
    'usedRange': '2y',
    'historicalDataPrice': [{'date': 1729479600,
      'open': 37,
      'high': 37.05,
      'low': 36.2,
      'close': 36.25,
      'volume': 28390000,
      'adjustedClose': 28.5375},
     {'date': 1729566000,
      'open': 36.2,
      'high': 36.3,
      'low': 35.99,
      'close': 36.11,
      'volume': 29144300,
      'adjustedClose': 28.4273},
     {'date': 1729652400,
      'open': 35.91,
      'high': 35.94,
      'low': 35.57,
      'close': 35.66,
      'volume': 30769100,
      'adjustedClose': 28.0731},
     {'date': 1729738800,
      'open': 35.85,
      'high': 36.03,
      'low': 35.56,
      'close': 35.9,
      'volume': 23058900,
      'adjustedClose': 28.262},
     {'date': 1729825200,
      'open': 35.96,
      'high': 36.49,
      'low': 35.85,
      'close': 36.15,
      'volume': 32567000,
      'adjustedClose': 28.4588}

In [8]:
df = pd.DataFrame(linhas)
df

,Simbolo,Data,Open,High,Low,Close
0,PETR4,21/10/2024,37.00,37.05,36.20,36.25
1,PETR4,22/10/2024,36.20,36.30,35.99,36.11
2,PETR4,23/10/2024,35.91,35.94,35.57,35.66
3,PETR4,24/10/2024,35.85,36.03,35.56,35.90
4,PETR4,25/10/2024,35.96,36.49,35.85,36.15
...,...,...,...,...,...,...
1983,ITUB4,14/10/2026,49.29,49.37,48.14,48.98
1984,ITUB4,15/10/2026,48.41,49.15,47.81,48.35
1985,ITUB4,16/10/2026,48.61,48.80,47.73,48.09
1986,ITUB4,19/10/2026,48.30,48.75,47.71,47.90


In [9]:
df.duplicated().sum()

np.int64(0)

In [ ]:
senha = os.environ["PASSWORD"]
host = os.environ["HOST"]
username = os.environ["USERNAME"]

supabase = create_engine(f'postgresql+psycopg2://{username}:{senha}@{host}:5432/postgres')

df.to_sql('hist_moeda', supabase, if_exists='replace', index=False)